# Step 2: Filter notebook

In [1]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point, box, LineString
import os
import folium
from folium import Choropleth, CircleMarker, GeoJson
import branca.colormap as cm
from IPython.display import display
pd.set_option('display.max_columns', None)

from network_connectivity import *

## Imports

#### Import des segments

In [2]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters : GG or GE, bike or walk")
territory = 'GG' # GG or GE
network = "bike"  # walk or bike

if territory == 'GG':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GG/step-1'
    output_step2_path='../../Data/output/GG/step-2'
    output_step3_path='../../Data/output/GG/step-3'


    save_path = '../../Data/output/GG/step-1'

if territory == 'GE':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GE/step-1'
    output_step2_path='../../Data/output/GE/step-2'
    output_step3_path='../../Data/output/GE/step-3'


    save_path = '../../Data/output/GE/step-1'

save_filtered_attributes = True

# Load segments GeoDataFrame (with 'segment_id')
print("Loading segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
segmented_net = segmented_net.to_crs(operation_crs)

# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):
    # Clean geometries first
    gdf["geometry"] = gdf["geometry"].apply(lambda geom: geom.buffer(0) if geom is not None and not geom.is_valid else geom)
    print("Geometries cleaned")

    if not save_filtered_attributes:
        print("Note : Save option is disabled.")
        return

    # Parse save formats: comma-separated list allowed
    raw = str(row.get('save_format', '') or '')
    formats = [f.strip().lower() for f in raw.split(',') if f.strip()]
    if not formats:
        formats = ['csv']  # default fallback

    for fmt in formats:
        try:
            if fmt == 'parquet':
                dirpath = f'{output_step2_path}/parquet_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_parquet(f"{dirpath}/{attribute}.parquet")
                print(f"Filtered data saved for attribute: {attribute} in format: parquet")

            elif fmt == 'gpkg' or fmt == 'geopackage':
                dirpath = f'{output_step2_path}/gpkg_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_file(f"{dirpath}/{attribute}.gpkg", driver="GPKG")
                print(f"Filtered data saved for attribute: {attribute} in format: gpkg")

            elif fmt == 'csv':
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                # to_csv may not preserve geometry consistently; keep original behavior
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Filtered data saved for attribute: {attribute} in format: csv")

            else:
                # Unknown format -> fallback to csv and warn
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Warning: Unknown save format '{fmt}' for attribute {attribute}. Data saved as csv.")
        except Exception as e:
            print(f"Error saving {attribute} as {fmt}: {e}")

Set parameters : GG or GE, bike or walk
Loading segments...


#### Import des attributs

In [6]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info.xlsx", sheet_name="attributs_info")
attributs_info = attributs_info[attributs_info['include_in_index'] != False]

attributs_info

,Class,meta_attribute,attribute,include_in_index,source_type_GE,source_path_GE,file_name_GE,source_type_GG,source_path_GG,file_name_GG,merge_rule_GG,initial_weight,class_weight,buffer_size,impact_attribut,geometry_type,how,value_column,clip,filter_column,filter_values,crs,save_format,Unnamed: 23,commentaire
0,comfort,temperature,temperature,True,sitg,GE/temperature,CLIMAT_TEMPERATURE_14H00_P1_2020/CLIMAT_TEMPER...,opendata,GG/temperature,20250629_102250.LST.tif,prefer_opendata,0.5,0.5,1.0,defavorable,point,raster,temperature,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN
1,attractivite,stationnement_velo,stationnement_velo,True,sitg,NaN,NaN,gpkg_layer,GG/osm,osm_attributes.gpkg |stationnement_velo,prefer_opendata,0.5,0.5,10.0,favorable,point,count,NaN,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN
2,attractivite,borne_reparation,borne_reparation,True,sitg,NaN,NaN,gpkg_layer,GG/osm,osm_attributes.gpkg |borne_reparation,prefer_opendata,0.5,0.5,10.0,favorable,point,count,NaN,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN
3,attractivite,location,location,True,sitg,NaN,NaN,gpkg_layer,GG/osm,osm_attributes.gpkg |location,prefer_opendata,0.5,0.5,10.0,favorable,point,count,NaN,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN
4,attractivite,amenite,amenite,True,sitg,NaN,NaN,gpkg_layer,GG/osm,osm_attributes.gpkg |amenite,prefer_opendata,0.5,0.5,10.0,favorable,point,count,NaN,10.0,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN
5,infrastructure,piste,piste,True,sitg,NaN,NaN,gpkg_layer,GG/osm,osm_attributes.gpkg |piste,prefer_opendata,0.5,0.5,10.0,favorable,point,count,NaN,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN
6,infrastructure,bande,bande,True,sitg,NaN,NaN,gpkg_layer,GG/osm,osm_attributes.gpkg |bande,prefer_opendata,0.5,0.5,10.0,favorable,point,count,NaN,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN
7,infrastructure,connectivite,connectivite,True,NaN,NaN,NaN,NaN,NaN,NaN,prefer_opendata,0.5,0.5,10.0,favorable,line,sum,conn_branching_in_buffer,NaN,filtered,1.0,2056.0,"parquet, csv, gpkg",NaN,NaN
8,NaN,largeur,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,peu d'info
9,NaN,revetement,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Connectivité du réseau**

In [7]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'connectivite'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
print(f"Processing attribute: {attribute}")

# Ensure geometry column is set to avoid spatial index errors
segmented_net = segmented_net.set_geometry("geometry")
print(segmented_net.columns)

# 1. Ajouter u, v, key si besoin
segmented_net = add_uv_columns(segmented_net)

# 2. Calculer les métriques et le score d’alternatives
segmented_net_index = compute_connectivity_metrics(
    segmented_net,
    buffer_m=35,
    compute_betweenness=False,
    betweenness_k=None,
    crs_meter_epsg=2056,
    main_metrics_only=False,
    conn_index_metric="conn_branching_in_buffer",
)

segmented_net_index['filtered'] = 1


# Preview
segmented_net_index.to_crs(target_crs).head()

Data initialized
Processing attribute: connectivite
Index(['u', 'v', 'key', 'infra_bike', 'bike_direction', 'source', 'highway',
       'cycleway', 'cycleway:left', 'cycleway:right', 'bicycle', 'oneway',
       'oneway:bicycle', 'name', 'maxspeed', 'len_m', 'edge_id', 'length',
       'surface', 'geometry', 'length_m', 'segment_id'],
      dtype='object')


,u,v,key,infra_bike,bike_direction,source,highway,cycleway,cycleway:left,cycleway:right,bicycle,oneway,oneway:bicycle,name,maxspeed,len_m,edge_id,length,surface,geometry,length_m,segment_id,conn_deadend_flag,conn_intersection_flag,conn_nodes_in_buffer,conn_intersections_in_buffer,conn_branching_in_buffer,conn_index_score,filtered
0,-9223359204931565246,-7580527903985510507,000000,bande_cyclable,both,OSM_highway_bike,residential,None,None,lane,None,None,None,Rue du 31-Décembre,None,33.141014,0,33.141014,None,"LINESTRING (6.15792 46.20527, 6.15782 46.20537...",33.141014,000000,False,True,8,5,0.28750,0.28750,1
1,-7580527903985510507,-9223359204931565246,000001,bande_cyclable,both,OSM_highway_bike,residential,None,None,lane,None,None,None,Rue du 31-Décembre,None,33.141014,0,33.141014,None,"LINESTRING (6.15767 46.20551, 6.15777 46.20542...",33.141014,000001,False,True,8,5,0.28750,0.28750,1
2,1073075817943591750,3073389916565780493,000002,sur_chaussée,both,OSM_highway_bike,tertiary,None,None,None,None,None,None,Route de la Vie de l'Etraz,50,228.789498,1,50.000000,None,"LINESTRING (5.95491 46.21124, 5.95495 46.21126...",228.789499,000002,False,True,10,3,0.23125,0.23125,1
3,3073389916565780493,-6413592105593826057,000003,sur_chaussée,both,OSM_highway_bike,tertiary,None,None,None,None,None,None,Route de la Vie de l'Etraz,50,228.789498,1,50.000000,None,"LINESTRING (5.95538 46.21155, 5.95577 46.21181...",228.789499,000003,False,False,9,0,0.10000,0.10000,1
4,-6413592105593826057,6086734632246958811,000004,sur_chaussée,both,OSM_highway_bike,tertiary,None,None,None,None,None,None,Route de la Vie de l'Etraz,50,228.789498,1,50.000000,None,"LINESTRING (5.95584 46.21186, 5.95624 46.21222)",228.789499,000004,False,False,10,1,0.13750,0.13750,1


In [9]:
save(save_filtered_attributes, row, segmented_net_index.to_crs(target_crs)[["segment_id", "infra_bike","geometry","conn_branching_in_buffer","conn_index_score","filtered"]], attribute)

Geometries cleaned
Filtered data saved for attribute: connectivite in format: parquet
Filtered data saved for attribute: connectivite in format: csv
Filtered data saved for attribute: connectivite in format: gpkg


In [10]:
segmented_net_index.to_crs(target_crs)[["key","segment_id", "infra_bike","geometry","conn_branching_in_buffer","conn_index_score","filtered"]]

,key,segment_id,infra_bike,geometry,conn_branching_in_buffer,conn_index_score,filtered
0,000000,000000,bande_cyclable,"LINESTRING (6.15792 46.20527, 6.15782 46.20537...",0.28750,0.28750,1
1,000001,000001,bande_cyclable,"LINESTRING (6.15767 46.20551, 6.15777 46.20542...",0.28750,0.28750,1
2,000002,000002,sur_chaussée,"LINESTRING (5.95491 46.21124, 5.95495 46.21126...",0.23125,0.23125,1
3,000003,000003,sur_chaussée,"LINESTRING (5.95538 46.21155, 5.95577 46.21181...",0.10000,0.10000,1
4,000004,000004,sur_chaussée,"LINESTRING (5.95584 46.21186, 5.95624 46.21222)",0.13750,0.13750,1
...,...,...,...,...,...,...,...
723146,723146,723146,sur_chaussée,"LINESTRING (6.18578 46.16687, 6.18573 46.16686...",0.25000,0.25000,1
723147,723147,723147,sur_chaussée,"LINESTRING (6.18567 46.16686, 6.18573 46.16686...",0.25000,0.25000,1
723148,723148,723148,sur_chaussée,"LINESTRING (6.48129 46.3704, 6.48134 46.37044,...",0.23125,0.23125,1
723149,723149,723149,sur_chaussée,"LINESTRING (6.48168 46.37072, 6.48146 46.37055...",0.23125,0.23125,1
